# HBB Mutation Classification — Part 2: Sequence-Based Model

This notebook demonstrates the **sequence-based pipeline**:

1. Parse real mutation coordinates from ClinVar's **Canonical SPDI** field
2. Fetch ±20 bp of reference sequence from **Ensembl REST API**
3. Apply the ALT allele to create mutated sequences
4. Encode sequences as **3-mer frequency vectors**
5. Train a Random Forest on the **delta k-mer features**

This moves beyond annotation labels toward true genomic sequence modelling.


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sequence_pipeline import (
    parse_spdi, accession_to_chrom, fetch_sequence,
    apply_mutation, kmer_vector, delta_kmer,
    ALL_3MER, WINDOW, K
)
from preprocess import load_clinvar, filter_hbb, make_binary_label

DATA_PATH = '../data/clinvar_result.txt'

## 1. Understand SPDI Notation

In [ ]:
example_spdi = 'NC_000011.10:5226966:A:T'
parsed = parse_spdi(example_spdi)
print(f'SPDI: {example_spdi}')
print(f'Accession: {parsed[0]}')
print(f'Position:  {parsed[1]}')
print(f'REF:       {parsed[2]}')
print(f'ALT:       {parsed[3]}')
print(f'Chromosome: {accession_to_chrom(parsed[0])}')

## 2. Fetch Reference Sequence from Ensembl

In [ ]:
accession, pos, ref, alt = parsed
chrom = accession_to_chrom(accession)

start = pos - WINDOW + 1
end   = pos + WINDOW + 1

ref_seq = fetch_sequence(chrom, start, end)
print(f'Window: chr{chrom}:{start}-{end}')
print(f'REF sequence ({len(ref_seq)} bp):')
print(ref_seq)
print(f'Center nucleotide (REF): {ref_seq[WINDOW]}')

## 3. Apply Mutation

In [ ]:
mut_seq = apply_mutation(ref_seq, ref, alt, WINDOW)
print('Reference:', ref_seq)
print('Mutated  :', mut_seq)
print(f'Change at position {WINDOW}: {ref} → {alt}')

## 4. K-mer Encoding

In [ ]:
ref_vec = kmer_vector(ref_seq)
mut_vec = kmer_vector(mut_seq)
delta   = delta_kmer(ref_seq, mut_seq)

print(f'K-mer feature dimension: {len(ref_vec)}')
print(f'All possible {K}-mers: {len(ALL_3MER)}')

# Show top shifted k-mers
kmer_df = pd.DataFrame({'k-mer': ALL_3MER, 'Δ frequency': delta})
kmer_df.reindex(kmer_df['Δ frequency'].abs().nlargest(10).index)

In [ ]:
# Visualise k-mer profile shift
non_zero = np.where((ref_vec > 0) | (mut_vec > 0))[0]
x = np.arange(len(non_zero))
labels = [ALL_3MER[i] for i in non_zero]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x - 0.2, ref_vec[non_zero], width=0.4, label='Reference', color='#2563EB', alpha=0.8)
ax.bar(x + 0.2, mut_vec[non_zero], width=0.4, label='Mutated',   color='#f87171', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_ylabel('Normalised Frequency')
ax.set_title('3-mer Profile: Reference vs Mutated Sequence')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Batch Sequence Fetching & Model Training

⚠️ **Note:** This step makes live API calls to Ensembl and may take several minutes.  
Run `python src/sequence_pipeline.py --max_variants 200` from the terminal instead for production use.

In [ ]:
import os
X_CACHE = '../results/sequence_X.npy'
y_CACHE = '../results/sequence_y.npy'

if os.path.exists(X_CACHE) and os.path.exists(y_CACHE):
    X = np.load(X_CACHE)
    y = np.load(y_CACHE)
    print(f'Loaded cached dataset: {X.shape}')
else:
    print('No cache found. Run src/sequence_pipeline.py first, or set RUN_LIVE=True below.')
    X, y = None, None

In [ ]:
if X is not None:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, roc_auc_score, roc_curve

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    print(f'Sequence Model – AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Benign','Pathogenic']))

## 6. Future Directions

| Improvement | Description |
|---|---|
| **DNABERT** | Transformer-based genomic language model for richer sequence embeddings |
| **Larger window** | Extend from ±20 to ±100+ bp to capture regulatory elements |
| **CNN/RNN** | Convolutional or recurrent models on raw sequence |
| **Multi-gene** | Extend pipeline to HBA1/HBA2 for alpha-thalassemia |
| **SHAP values** | Interpret which k-mer changes drive pathogenicity predictions |
